# Laptop Market Analysis - Data Extraction

## 1. Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import re
import html
import concurrent.futures

## 1. Extraction Implementation
1. **Request:** Send HTTP requests to collected URLs using the `requests` library.
2. **Parse:** Process the source code (HTML or hidden JSON objects like JSON-LD).
3. **Extract:** Locate and clean specific data components (Price, RAM, CPU, etc.).
4. **Standardize:** Map diverse site structures into a unified JSON format.
5. **Concurrency:** Use **Multithreading** to process multiple URLs simultaneously, significantly reducing execution time.

In [ ]:
# base scraper class that contains the common operations (generate id, clean html, fetch source)
# process_url: where to get the laptop's name, price, specs and social, structure will depend on the website

class BaseScraper:
    def __init__(self):
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        }

    def generate_id(self, url, prefix):
        # [website_name]_[laptop_name]
        slug = url.split('/')[-1].replace('.html', '').split('?')[0]
        return f"{prefix}_{slug}"

    def clean_html_text(self, text):
        if not text: return ""
        return BeautifulSoup(text, "html.parser").get_text(" ", strip=True)

    def fetch_html(self, url):
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            if response.status_code == 200:
                return response.text
            else:
                print(f"- Failed to load {url}: Status {response.status_code}")
                return None
        except Exception as e:
            print(f"- Connection error for {url}: {e}")
            return None

    def process_url(self, item):
        raise NotImplementedError("Child classes must implement process_url()")

In [ ]:
def run_scrape(scraper_instance, input_file, output_file, max_workers=5):
    try:
        with open(input_file, "r", encoding="utf-8") as f:
            url_list = json.load(f)
    except FileNotFoundError:
        print(f"Cant find {input_file}")
        return

    # begin scraping
    print(f"Starting Scrape for {len(url_list)} URLs")
    start_time = time.time()
    final_dataset = []
    
    def task(item):
        time.sleep(random.uniform(0.1, 0.5))
        return scraper_instance.process_url(item)

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {executor.submit(task, item): item for item in url_list}
        
        completed = 0
        for future in concurrent.futures.as_completed(future_to_url):
            completed += 1
            record = future.result()
            
            if record:
                final_dataset.append(record)
                
            if completed % 10 == 0 or completed == len(url_list):
                print(f"Progress: [{completed}/{len(url_list)}] items processed.")

    # save results
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(final_dataset, f, ensure_ascii=False, indent=4)
        
    end_time = time.time()
    print(f"\nScraped {len(final_dataset)} items in {round(end_time - start_time, 2)} seconds")
    print(f"📁 Data saved to: {output_file}\n")

## 4. Data Extraction

In [ ]:
class TGDDScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item['name']
        is_available = True
        current_price = 0
        specs = {}
        rating = 0.0
        review_count = 0
        satisfied_text = ""

        if soup.find("div", class_="notselling"):
            is_available = False

        # TGDT save most important info in the "productld" script
        ld_script = soup.find("script", id="productld")
        if ld_script:
            try:
                data = json.loads(ld_script.string)
                
                if isinstance(data, list):
                    data = data[0]
                
                name = data.get("name") or name
                
                offers = data.get("offers") or {}
                current_price = int(offers.get("price") or 0)
                
                aggregate = data.get("aggregateRating") or {}
                rating = float(aggregate.get("ratingValue") or 0.0)
                review_count = int(aggregate.get("reviewcount") or 0)
                
                # laptop specs
                additional_props = data.get("additionalProperty") or []
                for prop in additional_props:
                    if isinstance(prop, dict): 
                        key = prop.get("name")
                        raw_val = prop.get("value")
                        
                        if key and raw_val:
                            val = self.clean_html_text(str(raw_val))
                            specs[key] = val
                            
            except Exception as e:
                print(f"  [!] Error parsing JSON-LD for {url}: {e}")

        satisfied_tag = soup.find("p", class_="point-satisfied")
        if satisfied_tag:
            satisfied_text = satisfied_tag.get_text(strip=True)

        # Format output
        return {
            "id": self.generate_id(url, "tgdd"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": is_available
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": rating,
                "review_count": review_count,
                "satisfied_info": satisfied_text
            }
        }

In [ ]:
tgdd_scraper = TGDDScraper()
    
# Define inputs and outputs
# (Assuming the URL harvester saved the file as 'urls/01_tgdd_urls.json')
INPUT_FILE = "urls/01_tgdd_urls.json" 
OUTPUT_FILE = "data_tgdd.json"

# Run the multithreaded process using 5 workers (Safe limit for e-commerce)
run_scrape(
    scraper_instance=tgdd_scraper, 
    input_file=INPUT_FILE, 
    output_file=OUTPUT_FILE, 
    max_workers=15
)

--- Starting Deep Scrape for 583 URLs ---
  [!] Connection error for https://www.thegioididong.comlaptop-ai: HTTPSConnectionPool(host='www.thegioididong.comlaptop-ai', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='www.thegioididong.comlaptop-ai', port=443): Failed to resolve 'www.thegioididong.comlaptop-ai' ([Errno 8] nodename nor servname provided, or not known)"))
  [!] Connection error for https://www.thegioididong.comlaptop-ssd-3-tb: HTTPSConnectionPool(host='www.thegioididong.comlaptop-ssd-3-tb', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='www.thegioididong.comlaptop-ssd-3-tb', port=443): Failed to resolve 'www.thegioididong.comlaptop-ssd-3-tb' ([Errno 8] nodename nor servname provided, or not known)"))
  [!] Connection error for https://www.thegioididong.comlaptop-core-7: HTTPSConnectionPool(host='www.thegioididong.comlaptop-core-7', port=443): Max retries exceeded with url: 

In [ ]:
class PhongVuScraper(BaseScraper):
    # search through nested json dicts
    def recursive_find(self, data, target_key): 
        if isinstance(data, dict):
            if target_key in data:
                return data[target_key]
            for key, value in data.items():
                result = self.recursive_find(value, target_key)
                if result is not None:
                    return result
        elif isinstance(data, list):
            for item in data:
                result = self.recursive_find(item, target_key)
                if result is not None:
                    return result
        return None

    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item['name']
        current_price = 0
        specs = {}
        
        title_tag = soup.find("title")
        if title_tag:
            name = title_tag.get_text(strip=True).replace(" - Phong Vũ", "").strip()

        next_data_script = soup.find("script", id="__NEXT_DATA__")
        if next_data_script:
            try:
                json_data = json.loads(next_data_script.string)
                
                # price
                price_block = self.recursive_find(json_data, "priceAndPromotions")
                if price_block and isinstance(price_block, dict):
                    current_price = int(price_block.get("price") or 0)
                
                # specs
                short_desc = self.recursive_find(json_data, "shortDescription")
                if short_desc:
                    desc_soup = BeautifulSoup(short_desc, "html.parser")
                    
                    for br in desc_soup.find_all("br"):
                        br.replace_with("\n")
                        
                    lines = desc_soup.get_text().split("\n")
                    
                    for line in lines:
                        clean_line = line.strip().lstrip("-").strip()
                        if ":" in clean_line:
                            # split into key: value
                            parts = clean_line.split(":", 1)
                            key = parts[0].strip()
                            val = parts[1].strip()
                            specs[key] = val
                            
            except Exception as e:
                print(f"  [!] Error parsing JSON for {url}: {e}")

        # format output
        return {
            "id": self.generate_id(url, "phongvu"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": True if current_price > 0 else False
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": 0.0,    # Phong Vu has no social/rating data
                "review_count": 0,
                "satisfied_info": ""
            }
        }

In [ ]:

pv_scraper = PhongVuScraper()

run_scrape(
    scraper_instance=pv_scraper, 
    input_file="urls/01_phongvu_urls.json", 
    output_file="data_phongvu.json", 
    max_workers=2
)

--- Starting Deep Scrape for 464 URLs ---
Progress: [10/464] items processed.
Progress: [20/464] items processed.
Progress: [30/464] items processed.
Progress: [40/464] items processed.
Progress: [50/464] items processed.
Progress: [60/464] items processed.
Progress: [70/464] items processed.
Progress: [80/464] items processed.
  [!] Failed to load https://phongvu.vn//www.dmca.com/Protection/Status.aspx?ID=73ee7811-7aa7-44d0-bb06-6c0df0da41d8&refurl=https://phongvu.vn/c/laptop: Status 404
Progress: [90/464] items processed.
Progress: [100/464] items processed.
Progress: [110/464] items processed.
Progress: [120/464] items processed.
Progress: [130/464] items processed.
Progress: [140/464] items processed.
Progress: [150/464] items processed.
Progress: [160/464] items processed.
Progress: [170/464] items processed.
Progress: [180/464] items processed.
Progress: [190/464] items processed.
Progress: [200/464] items processed.
Progress: [210/464] items processed.
Progress: [220/464] items 

In [ ]:
class CellphoneSScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item['name']
        is_available = False 
        current_price = 0
        specs = {}
        rating = 0.0
        review_count = 0

        meta_keywords = soup.find("meta", attrs={"name": "keywords"})
        if meta_keywords and meta_keywords.get("content"):
            # name
            name = meta_keywords.get("content").split(',')[0].strip()

        scripts = soup.find_all("script", type="application/ld+json")
        for script in scripts:
            try:
                if not script.string:
                    continue
                    
                data = json.loads(script.string)
                
                if isinstance(data, list):
                    data = data[0]
                    
                if data.get("@type") == "Product":
                    
                    # price
                    offers = data.get("offers") or {}
                    raw_price = offers.get("price")
                    if raw_price:
                        current_price = int(raw_price)
                        is_available = True 
                        
                    # specs
                    additional_props = data.get("additionalProperty") or []
                    for prop in additional_props:
                        if isinstance(prop, dict):
                            key = prop.get("name")
                            val = prop.get("value")
                            if key and val:
                                specs[key] = self.clean_html_text(str(val))
                                
                    # ratings
                    reviews = data.get("review") or []
                    if isinstance(reviews, list) and len(reviews) > 0:
                        review_count = len(reviews)
                        total_score = 0
                        
                        for r in reviews:
                            if isinstance(r, dict):
                                r_rating = r.get("reviewRating") or {}
                                r_value = r_rating.get("ratingValue") or 0
                                total_score += float(r_value)
                                
                        rating = round(total_score / review_count, 1)
                        
                    break 
                    
            except Exception as e:
                pass 

        # format output
        return {
            "id": self.generate_id(url, "cps"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": is_available
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": rating,
                "review_count": review_count,
                "satisfied_info": "" # CellphoneS doesn't provide this metric
            }
        }



In [ ]:
cps_scraper = CellphoneSScraper()

# Run the multithreaded process 
run_scrape(
    scraper_instance=cps_scraper, 
    input_file="urls/01_cellphones_urls.json", 
    output_file="data_cellphones.json", 
    max_workers=2
)

--- Starting Deep Scrape for 657 URLs ---
Progress: [10/657] items processed.
Progress: [20/657] items processed.
Progress: [30/657] items processed.
Progress: [40/657] items processed.
Progress: [50/657] items processed.
Progress: [60/657] items processed.
Progress: [70/657] items processed.
Progress: [80/657] items processed.
Progress: [90/657] items processed.
Progress: [100/657] items processed.
Progress: [110/657] items processed.
Progress: [120/657] items processed.
Progress: [130/657] items processed.
Progress: [140/657] items processed.
Progress: [150/657] items processed.
Progress: [160/657] items processed.
Progress: [170/657] items processed.
Progress: [180/657] items processed.
Progress: [190/657] items processed.
Progress: [200/657] items processed.
Progress: [210/657] items processed.
Progress: [220/657] items processed.
Progress: [230/657] items processed.
Progress: [240/657] items processed.
Progress: [250/657] items processed.
Progress: [260/657] items processed.
Progr

In [ ]:
class AnPhatScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item['name']
        current_price = 0
        specs = {}
        review_count = 0
        view_count = 0

        # name
        name_tag = soup.find("h1", class_="js-product-name")
        if name_tag:
            name = name_tag.get_text(strip=True)

        # price
        price_tag = soup.find("b", class_="js-pro-total-price")
        if price_tag and price_tag.has_attr("data-price"):
            try:
                current_price = int(price_tag["data-price"])
            except ValueError:
                pass

        # specs
        summary_div = soup.find("div", class_="pro-info-summary")
        if summary_div:
            spec_items = summary_div.find_all("span", class_="item")
            for spec in spec_items:
                text = spec.get_text(strip=True)
                if ":" in text:
                    parts = text.split(":", 1)
                    key = parts[0].strip()
                    val = parts[1].strip()
                    specs[key] = val

        # reviews
        if name_tag:
            info_div = name_tag.find_next_sibling("div")
            if info_div:
                info_text = info_div.get_text(separator=" ", strip=True)
                
                rev_match = re.search(r'(\d+)\s*đánh giá', info_text)
                if rev_match:
                    review_count = int(rev_match.group(1))
                
                view_match = re.search(r'Lượt xem:\s*([\d\.]+)', info_text)
                if view_match:
                    clean_views = view_match.group(1).replace(".", "")
                    view_count = int(clean_views)

        # format output
        return {
            "id": self.generate_id(url, "anphat"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": True if current_price > 0 else False
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": 0.0, # An Phat doesn't show average star rating in the snippet
                "review_count": review_count,
                "satisfied_info": f"{view_count} Lượt xem" # We repurpose satisfied_info to store views!
            }
        }



In [ ]:
anphat_scraper = AnPhatScraper()

# Run the multithreaded process 
run_scrape(
    scraper_instance=anphat_scraper, 
    input_file="urls/01_anphat_urls.json", 
    output_file="data_anphat.json", 
    max_workers=5
)

--- Starting Deep Scrape for 1006 URLs ---
Progress: [10/1006] items processed.
Progress: [20/1006] items processed.
Progress: [30/1006] items processed.
Progress: [40/1006] items processed.
Progress: [50/1006] items processed.
Progress: [60/1006] items processed.
Progress: [70/1006] items processed.
Progress: [80/1006] items processed.
Progress: [90/1006] items processed.
Progress: [100/1006] items processed.
Progress: [110/1006] items processed.
Progress: [120/1006] items processed.
Progress: [130/1006] items processed.
Progress: [140/1006] items processed.
Progress: [150/1006] items processed.
Progress: [160/1006] items processed.
Progress: [170/1006] items processed.
Progress: [180/1006] items processed.
Progress: [190/1006] items processed.
Progress: [200/1006] items processed.
Progress: [210/1006] items processed.
Progress: [220/1006] items processed.
Progress: [230/1006] items processed.
Progress: [240/1006] items processed.
Progress: [250/1006] items processed.
Progress: [260/1

In [ ]:
class FPTScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item['name']
        is_available = False
        current_price = 0
        specs = {}
        rating = 0.0
        review_count = 0

        # FPT Shop store product info in a script like tgdt
        scripts = soup.find_all("script", type="application/ld+json")
        for script in scripts:
            if not script.string:
                continue
                
            try:
                data = json.loads(script.string)
                
                if isinstance(data, list):
                    data = data[0] if len(data) > 0 else {}

                if data.get("@type") == "Product":
                    
                    # name
                    name = data.get("name") or name
                    
                    # price
                    offers = data.get("offers") or {}
                    current_price = int(offers.get("price") or 0)
                    
                    # availability
                    availability_status = offers.get("availability", "")
                    if current_price > 0 or "InStock" in availability_status:
                        is_available = True
                    
                    # ratings
                    aggregate = data.get("aggregateRating") or {}
                    rating = float(aggregate.get("ratingValue") or 0.0)
                    review_count = int(aggregate.get("reviewCount") or 0)
                    
                    # specs
                    additional_props = data.get("additionalProperty") or []
                    for prop in additional_props:
                        if isinstance(prop, dict):
                            key = prop.get("name")
                            val = prop.get("value")
                            if key and val:
                                specs[key] = str(val).strip()
                    
                    break 

            except Exception as e:
                print(f"  [!] Error parsing JSON-LD for {url}: {e}")

        # format output
        return {
            "id": self.generate_id(url, "fpt"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": is_available
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": rating,
                "review_count": review_count,
                "satisfied_info": ""
            }
        }


In [ ]:
fpt_scraper = FPTScraper()

# Run the multithreaded process 
run_scrape(
    scraper_instance=fpt_scraper, 
    input_file="urls/01_fpt_urls.json", 
    output_file="data_fpt.json", 
    max_workers=5
)

--- Starting Deep Scrape for 418 URLs ---
Progress: [10/418] items processed.
Progress: [20/418] items processed.
Progress: [30/418] items processed.
Progress: [40/418] items processed.
Progress: [50/418] items processed.
Progress: [60/418] items processed.
Progress: [70/418] items processed.
Progress: [80/418] items processed.
Progress: [90/418] items processed.
Progress: [100/418] items processed.
Progress: [110/418] items processed.
Progress: [120/418] items processed.
Progress: [130/418] items processed.
Progress: [140/418] items processed.
Progress: [150/418] items processed.
Progress: [160/418] items processed.
Progress: [170/418] items processed.
Progress: [180/418] items processed.
Progress: [190/418] items processed.
Progress: [200/418] items processed.
Progress: [210/418] items processed.
Progress: [220/418] items processed.
Progress: [230/418] items processed.
Progress: [240/418] items processed.
Progress: [250/418] items processed.
Progress: [260/418] items processed.
Progr

In [ ]:
class GearVNScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        raw_html = self.fetch_html(url)
        if not raw_html:
            return None
            
        soup = BeautifulSoup(raw_html, 'html.parser')
        
        name = item['name']
        current_price = 0
        specs = {}
        rating = 0.0
        review_count = 0
        
        # name
        meta_title = soup.find("meta", property="og:title")
        if meta_title:
            name = meta_title.get("content", name)
            
        # price
        meta_price = soup.find("meta", property="og:price:amount") or soup.find("meta", property="product:price:amount")
        if meta_price:
            try:
                current_price = int(meta_price.get("content", 0))
            except ValueError:
                pass

        decoded_html = html.unescape(raw_html)
        decoded_soup = BeautifulSoup(decoded_html, 'html.parser')
        
        # specs
        for tr in decoded_soup.find_all("tr", class_="row-info"):
            tds = tr.find_all("td")
            if len(tds) >= 2:
                # Get text and clean up extra spaces
                key = tds[0].get_text(" ", strip=True)
                val = tds[1].get_text(" ", strip=True)
                
                if key and val:
                    specs[key] = val

        # ratings
        rating_div = soup.find("div", class_="product-reviews--number")
        if rating_div:
            rate_text = rating_div.get_text(strip=True).split('/')[0]
            try:
                rating = float(rate_text)
            except ValueError:
                pass
                
        review_total_div = soup.find("div", class_="product-reviews--total")
        if review_total_div:
            match = re.search(r'\d+', review_total_div.get_text())
            if match:
                review_count = int(match.group())

        # format output
        return {
            "id": self.generate_id(url, "gearvn"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": True if current_price > 0 else False
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": rating,
                "review_count": review_count,
                "satisfied_info": "" # GearVN doesn't provide a "satisfied customers" metric
            }
        }



In [ ]:
gearvn_scraper = GearVNScraper()

# Run the multithreaded process 
run_scrape(
    scraper_instance=gearvn_scraper, 
    input_file="urls/01_gearvn_urls.json", 
    output_file="data_gearvn.json", 
    max_workers=5
)

--- Starting Deep Scrape for 363 URLs ---
Progress: [10/363] items processed.
Progress: [20/363] items processed.
Progress: [30/363] items processed.
Progress: [40/363] items processed.
Progress: [50/363] items processed.
Progress: [60/363] items processed.
Progress: [70/363] items processed.
Progress: [80/363] items processed.
Progress: [90/363] items processed.
Progress: [100/363] items processed.
Progress: [110/363] items processed.
Progress: [120/363] items processed.
Progress: [130/363] items processed.
Progress: [140/363] items processed.
Progress: [150/363] items processed.
Progress: [160/363] items processed.
Progress: [170/363] items processed.
Progress: [180/363] items processed.
Progress: [190/363] items processed.
Progress: [200/363] items processed.
Progress: [210/363] items processed.
Progress: [220/363] items processed.
Progress: [230/363] items processed.
Progress: [240/363] items processed.
Progress: [250/363] items processed.
Progress: [260/363] items processed.
Progr

In [ ]:
class ThinkProScraper(BaseScraper):
    def process_url(self, item):
        url = item['url']
        html = self.fetch_html(url)
        if not html:
            return None
            
        soup = BeautifulSoup(html, 'html.parser')
        
        name = item.get('name', '')
        current_price = 0
        specs = {}
        review_count = 0
        rating = 0.0

        # name
        h1_tag = soup.find("h1")
        if h1_tag:
            name = h1_tag.get_text(strip=True)

        # price
        scripts = soup.find_all("script", type="application/ld+json")
        for script in scripts:
            try:
                data = json.loads(script.string)
            
                if isinstance(data, list):
                    for obj in data:
                        if "price" in obj:
                            current_price = int(obj.get("price", 0))
                            break
                        elif obj.get("@type") == "Product" and "offers" in obj:
                            current_price = int(obj["offers"].get("price", 0))
                            break
                elif isinstance(data, dict):
                    if "price" in data:
                        current_price = int(data.get("price", 0))
                    elif data.get("@type") == "Product" and "offers" in data:
                        current_price = int(data["offers"].get("price", 0))
            except Exception as e:
                pass 

        # specs
        rows = soup.find_all("tr")
        for row in rows:
            cols = row.find_all("td")
            if len(cols) == 2:
                key = cols[0].get_text(" ", strip=True)
                val = cols[1].get_text(" ", strip=True)
                
                if key and val:
                    specs[key] = val

        # ratings
        if "Chưa có đánh giá cho sản phẩm" not in html:
            review_count = 0 
            rating = 0.0

        # format output
        return {
            "id": self.generate_id(url, "thinkpro"),
            "metadata": {
                "source": item['source'],
                "crawl_date": time.strftime("%Y-%m-%dT%H:%M:%S"),
                "url": url
            },
            "product": {
                "name": name,
                "is_available": True if current_price > 0 else False
            },
            "pricing": {
                "current_price": current_price
            },
            "specs_raw": specs,
            "social": {
                "avg_rating": rating,
                "review_count": review_count,
                "satisfied_info": ""
            }
        }



In [ ]:
thinkpro_scraper = ThinkProScraper()

# Run the multithreaded process 
run_scrape(
    scraper_instance=thinkpro_scraper, 
    input_file="urls/01_thinkpro_urls.json", 
    output_file="data_thinkpro.json", 
    max_workers=5
)

--- Starting Deep Scrape for 356 URLs ---
Progress: [10/356] items processed.
  [!] Failed to load https://thinkpro.vn//www.dmca.com/Protection/Status.aspx?ID=a0c6f0b8-c3c6-4f8b-9344-9a817431c561&refurl=https://thinkpro.vn/laptop: Status 404
Progress: [20/356] items processed.
Progress: [30/356] items processed.
Progress: [40/356] items processed.
Progress: [50/356] items processed.
Progress: [60/356] items processed.
Progress: [70/356] items processed.
Progress: [80/356] items processed.
Progress: [90/356] items processed.
Progress: [100/356] items processed.
Progress: [110/356] items processed.
Progress: [120/356] items processed.
Progress: [130/356] items processed.
Progress: [140/356] items processed.
Progress: [150/356] items processed.
Progress: [160/356] items processed.
Progress: [170/356] items processed.
Progress: [180/356] items processed.
Progress: [190/356] items processed.
Progress: [200/356] items processed.
Progress: [210/356] items processed.
Progress: [220/356] items 